# Actividad 02b — Diagnóstico de SARIMA-Sutil

**Proyecto:** Predicción de Producción de Limón — reentrenamiento v2
**Fase:** 3 — Modelado (GC1)
**Motivo:** el experimento `exp_002_sarima_sutil` produjo métricas anómalas.

| Partición | MAE | RMSE | R² |
|---|---|---|---|
| train | 3,110.40 | 4,350.58 | 0.7073 |
| val   | 8,439.79 | 9,398.86 | **-0.3183** |
| test  | 8,139.30 | 9,128.16 | **-0.0969** |

El R² negativo en val y test significa que el modelo predice **peor que la media**
de la serie. Además el MAE de test (8,139) es **73% peor** que el baseline Naive
(4,704). El modelo ganador por AIC fue **SARIMA(1,1,3)(2,1,0,12)** (AIC=1874.93).

## Preguntas del diagnóstico

1. ¿Los residuos del ajuste en train son ruido blanco (Ljung-Box)?
2. ¿Las raíces AR/MA indican un modelo estable e invertible?
3. ¿Cómo se comporta la predicción en val (2024) y test (2025)?
4. ¿Un SARIMA más simple —(1,1,1)(1,1,0,12)— generaliza mejor?
5. ¿El problema es la serie o la selección de hiperparámetros?

> **Nota:** este notebook **no modifica** `exp_002_sarima_sutil`. Solo diagnostica.

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

SEED = 42
np.random.seed(SEED)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

PROC = 'v2_reentrenamiento/data/processed'
COL = 'produccion_t_sutil'
N_TRAIN, N_VAL, N_TEST = 90, 12, 12
P75_SUTIL = 23.2

# Configuracion de ajuste IDENTICA a la del experimento original (exp_002)
FIT_KW = dict(enforce_stationarity=False, enforce_invertibility=False,
              initialization='approximate_diffuse', trend='c')

ORDEN_GANADOR = ((1, 1, 3), (2, 1, 0, 12))   # seleccionado por AIC en exp_002
ORDEN_SIMPLE  = ((1, 1, 1), (1, 1, 0, 12))   # candidato parsimonioso

df = pd.read_csv(f'{PROC}/master_dataset_sutil_v2.csv', encoding='utf-8-sig')
df = df.rename(columns={df.columns[0]: 'anio'})
df['fecha'] = df['anio'].astype(str) + '-' + df['mes'].astype(str).str.zfill(2)
df = df.sort_values(['anio', 'mes']).reset_index(drop=True)

df['particion'] = 'buffer'
df.loc[(df.fecha >= '2016-07') & (df.fecha <= '2023-12'), 'particion'] = 'train'
df.loc[(df.fecha >= '2024-01') & (df.fecha <= '2024-12'), 'particion'] = 'val'
df.loc[df.fecha >= '2025-01', 'particion'] = 'test'

tr = df[df.particion == 'train'].reset_index(drop=True)
va = df[df.particion == 'val'].reset_index(drop=True)
te = df[df.particion == 'test'].reset_index(drop=True)
assert (len(tr), len(va), len(te)) == (N_TRAIN, N_VAL, N_TEST)

y_tr = tr[COL].astype(float).to_numpy()
y_va = va[COL].astype(float).to_numpy()
y_te = te[COL].astype(float).to_numpy()

print(f'train {len(tr)} ({tr.fecha.iloc[0]}..{tr.fecha.iloc[-1]}) | '
      f'val {len(va)} ({va.fecha.iloc[0]}..{va.fecha.iloc[-1]}) | '
      f'test {len(te)} ({te.fecha.iloc[0]}..{te.fecha.iloc[-1]})')

Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
train 90 (2016-07..2023-12) | val 12 (2024-01..2024-12) | test 12 (2025-01..2025-12)


## 0. Contexto de la serie — ¿hay quiebre estructural?

Antes de culpar al modelo, conviene ver el nivel de la serie por año.

In [2]:
niveles = df.groupby('anio')[COL].agg(['mean', 'std', 'min', 'max']).round(0)
niveles['cv_%'] = (100 * niveles['std'] / niveles['mean']).round(1)
print('NIVEL ANUAL DE LA SERIE SUTIL (toneladas)')
print(niveles.to_string())

m_tr = y_tr.mean(); m_va = y_va.mean(); m_te = y_te.mean()
print()
print(f'Media train (2016-07..2023-12): {m_tr:>10,.0f} t')
print(f'Media val   (2024):             {m_va:>10,.0f} t   ({100*(m_va/m_tr-1):+.1f}% vs train)')
print(f'Media test  (2025):             {m_te:>10,.0f} t   ({100*(m_te/m_tr-1):+.1f}% vs train)')
print()
print(f'Desv. est. train: {y_tr.std():,.0f} t  (CV = {100*y_tr.std()/m_tr:.1f}%)')

NIVEL ANUAL DE LA SERIE SUTIL (toneladas)
         mean     std      min      max  cv_%
anio                                         
2016  22394.0  4370.0  16714.0  29558.0  19.5
2017  13584.0  6315.0   4852.0  22283.0  46.5
2018  21509.0  6499.0  12762.0  29895.0  30.2
2019  24205.0  6927.0  12991.0  33259.0  28.6
2020  25132.0  6673.0  14810.0  35010.0  26.6
2021  27980.0  4362.0  22839.0  35180.0  15.6
2022  30558.0  8004.0  20730.0  41868.0  26.2
2023  23067.0  8319.0   9388.0  36552.0  36.1
2024  32209.0  8550.0  23811.0  44162.0  26.5
2025  31908.0  9103.0  18017.0  42120.0  28.5

Media train (2016-07..2023-12):     23,432 t
Media val   (2024):                 32,209 t   (+37.5% vs train)
Media test  (2025):                 31,908 t   (+36.2% vs train)

Desv. est. train: 8,042 t  (CV = 34.3%)


## PASO 1 — Diagnóstico de estabilidad del modelo ganador

### 1.1 Ajuste del modelo ganador SARIMA(1,1,3)(2,1,0,12)

In [3]:
def ajustar(y, orden, maxiter=500):
    ar, sr = orden
    return SARIMAX(y, order=ar, seasonal_order=sr, **FIT_KW).fit(disp=False, maxiter=maxiter)

fit_gan = ajustar(y_tr, ORDEN_GANADOR)
fit_sim = ajustar(y_tr, ORDEN_SIMPLE)

print('GANADOR  SARIMA(1,1,3)(2,1,0,12)  AIC =', round(fit_gan.aic, 2),
      '| BIC =', round(fit_gan.bic, 2), '| n_params =', len(fit_gan.params))
print('SIMPLE   SARIMA(1,1,1)(1,1,0,12)  AIC =', round(fit_sim.aic, 2),
      '| BIC =', round(fit_sim.bic, 2), '| n_params =', len(fit_sim.params))
print()
print('--- Coeficientes del modelo GANADOR ---')
print(pd.DataFrame({'coef': fit_gan.params, 'p_value': fit_gan.pvalues}).round(4).to_string())

GANADOR  SARIMA(1,1,3)(2,1,0,12)  AIC = 1874.93 | BIC = 1894.93 | n_params = 8
SIMPLE   SARIMA(1,1,1)(1,1,0,12)  AIC = 1886.52 | BIC = 1899.01 | n_params = 5

--- Coeficientes del modelo GANADOR ---
           coef  p_value
0 -9.754980e+01   0.1892
1  4.612000e-01   0.0495
2 -4.933000e-01   0.1416
3 -3.194000e-01   0.0533
4 -1.644000e-01   0.3855
5 -6.126000e-01   0.0000
6 -2.997000e-01   0.0103
7  1.530589e+07   0.0000


### 1.2 Test de Ljung-Box sobre los residuos

**H0:** los residuos NO están autocorrelacionados (son ruido blanco).
Un p-valor < 0.05 rechaza H0 → **queda estructura sin capturar**.

Se usan los residuos estandarizados descartando las primeras `d + D·s = 1 + 12 = 13`
observaciones, contaminadas por la inicialización difusa y la doble diferenciación.

In [4]:
D_TOTAL = 1 + 1 * 12   # d + D*s

def ljung(fit, nombre):
    resid = np.asarray(fit.standardized_forecasts_error[0])[D_TOTAL:]
    lb = acorr_ljungbox(resid, lags=[6, 12, 18, 24], return_df=True)
    lb.index.name = 'lag'
    lb = lb.rename(columns={'lb_stat': 'estadistico', 'lb_pvalue': 'p_valor'})
    lb['ruido_blanco'] = np.where(lb['p_valor'] > 0.05, 'SI (no rechaza H0)', 'NO (rechaza H0)')
    print(f'--- LJUNG-BOX | {nombre} | n_resid = {len(resid)} ---')
    print(lb.round(4).to_string())
    print()
    return lb

lb_gan = ljung(fit_gan, 'GANADOR (1,1,3)(2,1,0,12)')
lb_sim = ljung(fit_sim, 'SIMPLE  (1,1,1)(1,1,0,12)')

--- LJUNG-BOX | GANADOR (1,1,3)(2,1,0,12) | n_resid = 77 ---
     estadistico  p_valor        ruido_blanco
lag                                          
6         0.9671   0.9868  SI (no rechaza H0)
12        7.9893   0.7860  SI (no rechaza H0)
18       13.1452   0.7829  SI (no rechaza H0)
24       23.8539   0.4700  SI (no rechaza H0)

--- LJUNG-BOX | SIMPLE  (1,1,1)(1,1,0,12) | n_resid = 77 ---
     estadistico  p_valor        ruido_blanco
lag                                          
6         2.8842   0.8232  SI (no rechaza H0)
12       10.4913   0.5729  SI (no rechaza H0)
18       19.6046   0.3555  SI (no rechaza H0)
24       34.8790   0.0702  SI (no rechaza H0)



### 1.3 Raíces de los polinomios AR y MA

**Convención:** para el polinomio característico `φ(L) = 0`, el modelo es
estacionario/invertible si **todas las raíces caen FUERA del círculo unitario**
(`|raíz| > 1`), equivalente a que las raíces inversas caigan dentro (`|1/raíz| < 1`).

Una raíz con `|raíz| ≤ 1` indica un modelo **no estacionario / no invertible**,
que puede producir pronósticos explosivos o degenerados. El grid original usó
`enforce_stationarity=False` y `enforce_invertibility=False`, así que nada impidió
seleccionar un modelo inestable.

In [5]:
def raices(fit, nombre):
    print(f'=== RAICES | {nombre} ===')
    for etq, r in [('AR (incl. estacional)', fit.arroots), ('MA (incl. estacional)', fit.maroots)]:
        r = np.asarray(r)
        if r.size == 0:
            print(f'  {etq}: sin raices (polinomio de grado 0)')
            continue
        mod = np.abs(r)
        n_mal = int((mod <= 1.0).sum())
        print(f'  {etq}: {len(r)} raices | |raiz| min = {mod.min():.4f} | '
              f'fuera del circulo (|r|>1): {len(r)-n_mal}/{len(r)} | '
              f'DENTRO o SOBRE (|r|<=1): {n_mal}')
        orden_idx = np.argsort(mod)[:5]
        for i in orden_idx:
            flag = '  <-- PROBLEMA (|r|<=1)' if mod[i] <= 1.0 else ''
            print(f'      raiz = {r[i].real:+.4f}{r[i].imag:+.4f}j | |raiz| = {mod[i]:.4f} | '
                  f'|1/raiz| = {1/mod[i]:.4f}{flag}')
        if n_mal > 0:
            print(f'      >> VEREDICTO {etq.split()[0]}: NO {"estacionario" if "AR" in etq else "invertible"}')
        else:
            print(f'      >> VEREDICTO {etq.split()[0]}: OK')
    print()

raices(fit_gan, 'GANADOR (1,1,3)(2,1,0,12)')
raices(fit_sim, 'SIMPLE  (1,1,1)(1,1,0,12)')

=== RAICES | GANADOR (1,1,3)(2,1,0,12) ===
  AR (incl. estacional): 25 raices | |raiz| min = 1.0515 | fuera del circulo (|r|>1): 25/25 | DENTRO o SOBRE (|r|<=1): 0
      raiz = +0.3538+0.9902j | |raiz| = 1.0515 | |1/raiz| = 0.9510
      raiz = +0.3538-0.9902j | |raiz| = 1.0515 | |1/raiz| = 0.9510
      raiz = -0.8015+0.6806j | |raiz| = 1.0515 | |1/raiz| = 0.9510
      raiz = -0.8015-0.6806j | |raiz| = 1.0515 | |1/raiz| = 0.9510
      raiz = +0.8015-0.6806j | |raiz| = 1.0515 | |1/raiz| = 0.9510
      >> VEREDICTO AR: OK
  MA (incl. estacional): 3 raices | |raiz| min = 1.0140 | fuera del circulo (|r|>1): 3/3 | DENTRO o SOBRE (|r|<=1): 0
      raiz = +1.0140-0.0000j | |raiz| = 1.0140 | |1/raiz| = 0.9862
      raiz = -1.4781-1.9525j | |raiz| = 2.4489 | |1/raiz| = 0.4083
      raiz = -1.4781+1.9525j | |raiz| = 2.4489 | |1/raiz| = 0.4083
      >> VEREDICTO MA: OK

=== RAICES | SIMPLE  (1,1,1)(1,1,0,12) ===
  AR (incl. estacional): 13 raices | |raiz| min = 1.0643 | fuera del circulo (|r|>1): 

### 1.4 Predicciones en val (2024) y test (2025)

> 🔴 **Hallazgo previo — bug de horizonte en `exp_002`.**
> El notebook original (`02_gc1_sarima_prophet.ipynb`) calcula:
> ```python
> pred_val  = fit.forecast(steps=N_VAL)    # 12 pasos desde fin de train -> 2024
> pred_test = fit.forecast(steps=N_TEST)   # 12 pasos desde fin de train -> 2024 OTRA VEZ
> ```
> Ambas llamadas parten del mismo fin de train, así que **la predicción de test
> (2025) es en realidad el pronóstico de 2024 reutilizado**. Se verificó contra
> `exp_002_sarima_sutil/predicciones.csv`: `pred_val` y `pred_test` son idénticas
> (diferencia máxima = 0.0).
>
> El pronóstico correcto para 2025 requiere 24 pasos y tomar los últimos 12.
> Aquí se evalúan **ambas variantes** para separar el efecto del bug del efecto
> del modelo.

In [6]:
def metricas(y, p):
    return {'mae': float(mean_absolute_error(y, p)),
            'rmse': float(np.sqrt(mean_squared_error(y, p))),
            'r2': float(r2_score(y, p))}

def predicciones(fit):
    f24 = np.asarray(fit.forecast(steps=N_VAL + N_TEST))
    return {'pred_train': np.asarray(fit.predict(start=0, end=N_TRAIN - 1)),
            'pred_val': f24[:N_VAL],
            'test_bug': np.asarray(fit.forecast(steps=N_TEST)),   # replica exp_002
            'test_ok': f24[N_VAL:]}                                # horizonte correcto

P_gan = predicciones(fit_gan)
P_sim = predicciones(fit_sim)

print('Verificacion del bug replicado:',
      'pred_val == test_bug ?', np.allclose(P_gan['pred_val'], P_gan['test_bug']))
print()

comp = pd.DataFrame({
    'fecha_val': va.fecha.values, 'real_2024': y_va,
    'pred_val': P_gan['pred_val'].round(1),
    'fecha_test': te.fecha.values, 'real_2025': y_te,
    'pred_test_BUG': P_gan['test_bug'].round(1),
    'pred_test_OK': P_gan['test_ok'].round(1),
})
print('GANADOR — predicho vs real')
print(comp.to_string(index=False))

Verificacion del bug replicado: pred_val == test_bug ? True

GANADOR — predicho vs real
fecha_val  real_2024  pred_val fecha_test  real_2025  pred_test_BUG  pred_test_OK
  2024-01  36531.883   33702.7    2025-01  38325.524        33702.7       30058.2
  2024-02  43992.447   35871.1    2025-02  40092.876        35871.1       32989.0
  2024-03  44161.843   33276.7    2025-03  42119.744        33276.7       29985.4
  2024-04  43176.231   28252.3    2025-04  41178.818        28252.3       24677.4
  2024-05  39200.488   25338.2    2025-05  37552.279        25338.2       21917.4
  2024-06  30460.085   19284.7    2025-06  34500.774        19284.7       15812.2
  2024-07  24936.369   16703.3    2025-07  23417.927        16703.3       12614.6
  2024-08  25176.553   13864.9    2025-08  18017.175        13864.9        9203.7
  2024-09  24222.320   15757.2    2025-09  18304.830        15757.2       11416.1
  2024-10  23811.288   17027.9    2025-10  21230.276        17027.9       11788.4
  2024-11 

In [7]:
def perfil(nombre, y, p):
    err = p - y
    print(f'  {nombre:22s} media_pred={p.mean():>9,.0f} | media_real={y.mean():>9,.0f} | '
          f'sesgo={err.mean():>+9,.0f} ({100*err.mean()/y.mean():>+6.1f}%) | '
          f'rango_pred=[{p.min():>8,.0f}, {p.max():>8,.0f}] | sd_pred={p.std():>8,.0f}')

print('PERFIL DE LAS PREDICCIONES (GANADOR)')
perfil('val 2024', y_va, P_gan['pred_val'])
perfil('test 2025 (BUG)', y_te, P_gan['test_bug'])
perfil('test 2025 (OK)', y_te, P_gan['test_ok'])
print()
print('  sd de la serie real: train=%.0f | val=%.0f | test=%.0f' % (y_tr.std(), y_va.std(), y_te.std()))
print()
print('Diagnostico de forma del pronostico a 24 pasos (GANADOR):')
f24 = np.concatenate([P_gan['pred_val'], P_gan['test_ok']])
print('  primeros 12:', np.round(f24[:12], 0))
print('  ultimos 12 :', np.round(f24[12:], 0))
print(f'  amplitud (max-min) primeros 12: {np.ptp(f24[:12]):,.0f} | ultimos 12: {np.ptp(f24[12:]):,.0f}')
print(f'  nivel medio primeros 12: {f24[:12].mean():,.0f} | ultimos 12: {f24[12:].mean():,.0f}')

PERFIL DE LAS PREDICCIONES (GANADOR)
  val 2024               media_pred=   23,769 | media_real=   32,209 | sesgo=   -8,440 ( -26.2%) | rango_pred=[  13,865,   35,871] | sd_pred=   7,353
  test 2025 (BUG)        media_pred=   23,769 | media_real=   31,908 | sesgo=   -8,139 ( -25.5%) | rango_pred=[  13,865,   35,871] | sd_pred=   7,353
  test 2025 (OK)         media_pred=   19,877 | media_real=   31,908 | sesgo=  -12,032 ( -37.7%) | rango_pred=[   9,204,   32,989] | sd_pred=   7,865

  sd de la serie real: train=8042 | val=8186 | test=8716

Diagnostico de forma del pronostico a 24 pasos (GANADOR):
  primeros 12: [33703. 35871. 33277. 28252. 25338. 19285. 16703. 13865. 15757. 17028.
 20624. 25524.]
  ultimos 12 : [30058. 32989. 29985. 24677. 21917. 15812. 12615.  9204. 11416. 11788.
 16228. 21829.]
  amplitud (max-min) primeros 12: 22,006 | ultimos 12: 23,785
  nivel medio primeros 12: 23,769 | ultimos 12: 19,877


## PASO 2 — Comparación contra un SARIMA más simple

Se compara el ganador por AIC contra **SARIMA(1,1,1)(1,1,0,12)** ajustado sobre el
mismo train, con configuración idéntica. Se añade el baseline **Naive** (y_t = y_{t-1})
como referencia, y una variante diagnóstica de **refit sobre train+val** para separar
"malos hiperparámetros" de "train demasiado antiguo".

In [8]:
# Baseline Naive sobre las mismas particiones
serie_full = df[COL].astype(float).to_numpy()
idx_val = df.index[df.particion == 'val'].to_numpy()
idx_test = df.index[df.particion == 'test'].to_numpy()
naive_val = serie_full[idx_val - 1]
naive_test = serie_full[idx_test - 1]

# Variante diagnostica: refit sobre train+val -> pronostico de test a 12 pasos
y_trval = np.concatenate([y_tr, y_va])
fit_gan_tv = ajustar(y_trval, ORDEN_GANADOR)
fit_sim_tv = ajustar(y_trval, ORDEN_SIMPLE)
pred_test_gan_tv = np.asarray(fit_gan_tv.forecast(steps=N_TEST))
pred_test_sim_tv = np.asarray(fit_sim_tv.forecast(steps=N_TEST))

filas = []
def fila(modelo, particion, y, p, nota=''):
    m = metricas(y, p)
    filas.append({'modelo': modelo, 'particion': particion, 'MAE': round(m['mae'], 2),
                  'RMSE': round(m['rmse'], 2), 'R2': round(m['r2'], 4), 'nota': nota})

fila('Naive', 'val', y_va, naive_val, 'baseline')
fila('Naive', 'test', y_te, naive_test, 'baseline')
fila('SARIMA GANADOR (1,1,3)(2,1,0,12)', 'train', y_tr, P_gan['pred_train'], '')
fila('SARIMA GANADOR (1,1,3)(2,1,0,12)', 'val', y_va, P_gan['pred_val'], '')
fila('SARIMA GANADOR (1,1,3)(2,1,0,12)', 'test', y_te, P_gan['test_bug'], 'horizonte BUG (replica exp_002)')
fila('SARIMA GANADOR (1,1,3)(2,1,0,12)', 'test', y_te, P_gan['test_ok'], 'horizonte CORREGIDO (24 pasos)')
fila('SARIMA SIMPLE (1,1,1)(1,1,0,12)', 'train', y_tr, P_sim['pred_train'], '')
fila('SARIMA SIMPLE (1,1,1)(1,1,0,12)', 'val', y_va, P_sim['pred_val'], '')
fila('SARIMA SIMPLE (1,1,1)(1,1,0,12)', 'test', y_te, P_sim['test_bug'], 'horizonte BUG')
fila('SARIMA SIMPLE (1,1,1)(1,1,0,12)', 'test', y_te, P_sim['test_ok'], 'horizonte CORREGIDO (24 pasos)')
fila('SARIMA GANADOR refit train+val', 'test', y_te, pred_test_gan_tv, 'DIAGNOSTICO (fuera de protocolo)')
fila('SARIMA SIMPLE refit train+val', 'test', y_te, pred_test_sim_tv, 'DIAGNOSTICO (fuera de protocolo)')

tabla = pd.DataFrame(filas)
print('COMPARATIVA COMPLETA — SUTIL')
print(tabla.to_string(index=False))

COMPARATIVA COMPLETA — SUTIL
                          modelo particion      MAE     RMSE      R2                             nota
                           Naive       val  3078.13  4269.75  0.7279                         baseline
                           Naive      test  4704.29  6317.41  0.4746                         baseline
SARIMA GANADOR (1,1,3)(2,1,0,12)     train  3110.40  4350.58  0.7073                                 
SARIMA GANADOR (1,1,3)(2,1,0,12)       val  8439.79  9398.86 -0.3183                                 
SARIMA GANADOR (1,1,3)(2,1,0,12)      test  8139.30  9128.16 -0.0969  horizonte BUG (replica exp_002)
SARIMA GANADOR (1,1,3)(2,1,0,12)      test 12031.69 12664.83 -1.1115   horizonte CORREGIDO (24 pasos)
 SARIMA SIMPLE (1,1,1)(1,1,0,12)     train  3338.29  4659.99  0.6642                                 
 SARIMA SIMPLE (1,1,1)(1,1,0,12)       val  4824.13  5607.45  0.5308                                 
 SARIMA SIMPLE (1,1,1)(1,1,0,12)      test  3595.00  

In [9]:
print('COMPARACION DIRECTA GANADOR vs SIMPLE (mismo horizonte)')
print()
for part, y, kg, ks in [('val 2024', y_va, 'pred_val', 'pred_val'),
                        ('test 2025 (corregido)', y_te, 'test_ok', 'test_ok')]:
    mg = metricas(y, P_gan[kg]); ms = metricas(y, P_sim[ks])
    mejor = 'SIMPLE' if ms['mae'] < mg['mae'] else 'GANADOR'
    delta = 100 * (ms['mae'] - mg['mae']) / mg['mae']
    print(f'  {part}:')
    print(f'    GANADOR  MAE={mg["mae"]:>10,.2f}  R2={mg["r2"]:>8.4f}')
    print(f'    SIMPLE   MAE={ms["mae"]:>10,.2f}  R2={ms["r2"]:>8.4f}   ({delta:+.1f}% vs ganador)')
    print(f'    >> mejor: {mejor}')
    print()

print('Referencia Naive:  MAE_val=%.2f | MAE_test=%.2f' % (
    mean_absolute_error(y_va, naive_val), mean_absolute_error(y_te, naive_test)))

COMPARACION DIRECTA GANADOR vs SIMPLE (mismo horizonte)

  val 2024:
    GANADOR  MAE=  8,439.79  R2= -0.3183
    SIMPLE   MAE=  4,824.13  R2=  0.5308   (-42.8% vs ganador)
    >> mejor: SIMPLE

  test 2025 (corregido):
    GANADOR  MAE= 12,031.69  R2= -1.1115
    SIMPLE   MAE=  3,638.14  R2=  0.7552   (-69.8% vs ganador)
    >> mejor: SIMPLE

Referencia Naive:  MAE_val=3078.13 | MAE_test=4704.29


### Descomposición del error: ¿sesgo de nivel o error de forma?

Si el error es dominado por **sesgo** (el modelo predice sistemáticamente bajo),
el problema es el quiebre de nivel de la serie, no la forma estacional. Si domina
la **varianza**, el problema es que el modelo no captura el patrón.

In [10]:
def descomponer(nombre, y, p):
    err = p - y
    sesgo = err.mean()
    mse = np.mean(err ** 2)
    var_err = np.var(err)
    print(f'  {nombre:34s} MSE={mse:>14,.0f} | sesgo^2={sesgo**2:>14,.0f} '
          f'({100*sesgo**2/mse:>5.1f}%) | varianza={var_err:>14,.0f} ({100*var_err/mse:>5.1f}%)')

print('DESCOMPOSICION DEL MSE  (sesgo^2 + varianza)')
descomponer('GANADOR val 2024', y_va, P_gan['pred_val'])
descomponer('GANADOR test 2025 (corregido)', y_te, P_gan['test_ok'])
descomponer('SIMPLE  val 2024', y_va, P_sim['pred_val'])
descomponer('SIMPLE  test 2025 (corregido)', y_te, P_sim['test_ok'])
descomponer('GANADOR refit train+val, test', y_te, pred_test_gan_tv)
descomponer('SIMPLE  refit train+val, test', y_te, pred_test_sim_tv)
print()
print('Correlacion pred vs real (captura de forma estacional):')
for n, p in [('GANADOR val', P_gan['pred_val']), ('GANADOR test_ok', P_gan['test_ok']),
             ('SIMPLE val', P_sim['pred_val']), ('SIMPLE test_ok', P_sim['test_ok'])]:
    y = y_va if 'val' in n else y_te
    print(f'  {n:20s} r = {np.corrcoef(y, p)[0,1]:+.4f}')

DESCOMPOSICION DEL MSE  (sesgo^2 + varianza)
  GANADOR val 2024                   MSE=    88,338,542 | sesgo^2=    71,230,126 ( 80.6%) | varianza=    17,108,416 ( 19.4%)
  GANADOR test 2025 (corregido)      MSE=   160,398,002 | sesgo^2=   144,761,613 ( 90.3%) | varianza=    15,636,389 (  9.7%)
  SIMPLE  val 2024                   MSE=    31,443,529 | sesgo^2=     8,539,902 ( 27.2%) | varianza=    22,903,626 ( 72.8%)
  SIMPLE  test 2025 (corregido)      MSE=    18,595,402 | sesgo^2=       275,279 (  1.5%) | varianza=    18,320,123 ( 98.5%)
  GANADOR refit train+val, test      MSE=    42,507,739 | sesgo^2=    29,801,672 ( 70.1%) | varianza=    12,706,067 ( 29.9%)
  SIMPLE  refit train+val, test      MSE=    76,610,965 | sesgo^2=    65,009,099 ( 84.9%) | varianza=    11,601,867 ( 15.1%)

Correlacion pred vs real (captura de forma estacional):
  GANADOR val          r = +0.8636
  GANADOR test_ok      r = +0.8912
  SIMPLE val           r = +0.8128
  SIMPLE test_ok       r = +0.8732


## PASO 3 — Métricas del candidato para registro

Si el modelo simple resulta mejor, estas son las métricas que irían a
`exp_002b_sarima_sutil_simple`. Se calculan con el **mismo protocolo** que
`exp_002` (incluida la máscara de shocks P75 sobre test) para que las filas del
`REGISTRO_MAESTRO.csv` sean comparables.

In [11]:
def shocks_test(df_all, pred_test_arr):
    aux = df_all.copy()
    var_pct = 100 * aux[COL].pct_change().abs()
    mask_test = (aux.particion == 'test').to_numpy()
    aux['shock'] = False
    aux.loc[mask_test, 'shock'] = (var_pct > P75_SUTIL)[mask_test].to_numpy()
    t = aux[mask_test].copy()
    t['predicho_t'] = pred_test_arr
    sh = t[t['shock']]
    mae_glob = float(mean_absolute_error(t[COL], t['predicho_t']))
    if len(sh) > 0:
        mae_sh = float(mean_absolute_error(sh[COL], sh['predicho_t']))
        ds = 100.0 * (mae_sh - mae_glob) / mae_glob
    else:
        mae_sh, ds = float('nan'), float('nan')
    return int(len(sh)), mae_sh, ds, sh.fecha.tolist()

for nombre, P, fit in [('GANADOR', P_gan, fit_gan), ('SIMPLE', P_sim, fit_sim)]:
    for etq, pt in [('horizonte BUG', P['test_bug']), ('horizonte CORREGIDO', P['test_ok'])]:
        n_sh, mae_sh, ds, fechas = shocks_test(df, pt)
        print(f'{nombre:8s} | {etq:20s} | n_shock={n_sh} | mae_shock={mae_sh:>10,.2f} | '
              f'delta_s={ds:>+7.2f}% | meses={fechas}')
print()

for nombre, fit, P in [('GANADOR (1,1,3)(2,1,0,12)', fit_gan, P_gan),
                       ('SIMPLE  (1,1,1)(1,1,0,12)', fit_sim, P_sim)]:
    print(f'--- {nombre} | metricas completas (horizonte CORREGIDO) ---')
    for part, y, p in [('train', y_tr, P['pred_train']), ('val', y_va, P['pred_val']),
                       ('test', y_te, P['test_ok'])]:
        m = metricas(y, p)
        print(f'   {part:5s} MAE={m["mae"]:>10,.2f} | RMSE={m["rmse"]:>10,.2f} | R2={m["r2"]:>8.4f}')
    print(f'   AIC={fit.aic:.2f} | BIC={fit.bic:.2f}')
    print()

GANADOR  | horizonte BUG        | n_shock=3 | mae_shock=  8,023.51 | delta_s=  -1.42% | meses=['2025-01', '2025-07', '2025-11']
GANADOR  | horizonte CORREGIDO  | n_shock=3 | mae_shock= 12,066.72 | delta_s=  +0.29% | meses=['2025-01', '2025-07', '2025-11']
SIMPLE   | horizonte BUG        | n_shock=3 | mae_shock=  2,794.68 | delta_s= -22.26% | meses=['2025-01', '2025-07', '2025-11']
SIMPLE   | horizonte CORREGIDO  | n_shock=3 | mae_shock=    755.01 | delta_s= -79.25% | meses=['2025-01', '2025-07', '2025-11']

--- GANADOR (1,1,3)(2,1,0,12) | metricas completas (horizonte CORREGIDO) ---
   train MAE=  3,110.40 | RMSE=  4,350.58 | R2=  0.7073
   val   MAE=  8,439.79 | RMSE=  9,398.86 | R2= -0.3183
   test  MAE= 12,031.69 | RMSE= 12,664.83 | R2= -1.1115
   AIC=1874.93 | BIC=1894.93

--- SIMPLE  (1,1,1)(1,1,0,12) | metricas completas (horizonte CORREGIDO) ---
   train MAE=  3,338.29 | RMSE=  4,659.99 | R2=  0.6642
   val   MAE=  4,824.13 | RMSE=  5,607.45 | R2=  0.5308
   test  MAE=  3,638.14

---

# Conclusión del diagnóstico

## Respuesta a la pregunta central

**El problema NO es que la serie de Limón Sutil sea intrínsecamente inmodelable con
SARIMA. Es un fallo de selección de hiperparámetros, corregible, agravado por un bug
de horizonte en el código de evaluación.**

## Evidencia

**1. El ajuste en train es correcto.** El test de Ljung-Box no rechaza H0 en ningún
lag (p = 0.99, 0.79, 0.78, 0.47 para lags 6/12/18/24): los residuos son ruido blanco,
no queda estructura sin capturar. El fallo no es de sub-ajuste.

**2. El modelo es estable e invertible.** Las 25 raíces AR y las 3 raíces MA caen
fuera del círculo unitario. No hay comportamiento explosivo. Sin embargo, la raíz MA
mínima (|r| = 1.0140) está pegada al borde de la no invertibilidad — señal coherente
con sobreparametrización. El grid original usó `enforce_stationarity=False` y
`enforce_invertibility=False`, de modo que nada habría impedido seleccionar un modelo
en la frontera.

**3. La predicción no oscila ni explota: se hunde de nivel.** La correlación entre
predicho y real es alta (r = +0.89 en test), es decir el modelo **sí captura la forma
estacional**. Lo que falla es el nivel: sesgo de −26.2% en val y −37.7% en test con
horizonte corregido. La descomposición del MSE lo confirma: **90.3% del error de test
es sesgo², solo 9.7% varianza**.

**4. Un modelo más simple generaliza mucho mejor.**

| Modelo | MAE_val | R²_val | MAE_test | R²_test |
|---|---|---|---|---|
| Ganador por AIC (1,1,3)(2,1,0,12) | 8,439.79 | −0.3183 | 12,031.69 | −1.1115 |
| **Simple (1,1,1)(1,1,0,12)** | **4,824.13** | **+0.5308** | **3,638.14** | **+0.7552** |
| Naive (baseline) | 3,078.13 | 0.7279 | 4,704.29 | 0.4746 |

El modelo simple reduce el MAE de test en **69.8%** y es el único SARIMA que
**supera al Naive en test** (−22.7%). Su sesgo² es apenas el 1.5% del MSE: es
prácticamente insesgado. En los meses de shock su ventaja se amplía
(MAE_shock = 755.01, Δs = −79.25%).

## Por qué falló la selección

El AIC del ganador (1874.93) es mejor que el del simple (1886.52), y el BIC también
(1894.93 vs 1899.01). El protocolo filtraba **top-5 por AIC** y solo después elegía
por MAE_val; con 11.6 puntos de AIC de desventaja, el modelo simple casi con
seguridad nunca llegó a la etapa de validación. **El pre-filtro por AIC envenenó la
selección**: premia el ajuste en train, donde el modelo de 8 parámetros gana, y
descarta al que generaliza.

## El mecanismo de fondo

La serie de Sutil crece de forma sostenida (~+4%/año: +2.0% de área cultivada y
+1.8% de rendimiento) y sufre caídas de ~25% en años de shock climático (2017 El Niño
Costero, 2023 Ciclón Yaku). La media de train (23,432 t) queda muy por debajo del
nivel de 2024-2025 (~32,000 t) por dos razones acumuladas: los 24 meses de shock
dentro de train (27% de la ventana) y la tendencia secular a lo largo de 7 años.

Con `trend='c'` y `d=1`, el modelo de 8 parámetros tiene grados de libertad
suficientes para extrapolar una **deriva descendente** que no existe. El modelo de 5
parámetros no puede inventarla y se mantiene plano — que en este caso resulta ser lo
correcto.

## Bug de horizonte detectado (afecta a exp_002 de ambos cultivos)

El notebook `02_gc1_sarima_prophet.ipynb` calculaba:

```python
pred_val  = fit.forecast(steps=N_VAL)    # pasos 1..12 desde fin de train -> 2024
pred_test = fit.forecast(steps=N_TEST)   # pasos 1..12 desde el MISMO punto -> 2024 otra vez
```

Ambas llamadas parten del fin de train, por lo que **la predicción de test (2025) era
el pronóstico de 2024 reutilizado**. Verificado sobre los CSV guardados: `pred_val` y
`pred_test` idénticas (diferencia máxima = 0.00) en `exp_002_sarima_sutil` y
`exp_002_sarima_dulce`. Los experimentos Naive y Prophet no están afectados.

El horizonte correcto para test requiere 24 pasos y tomar los últimos 12. Corregido,
el MAE_test del ganador empeora de 8,139.30 a **12,031.69**: el bug estaba
*enmascarando* la magnitud real del fallo.

## Decisiones derivadas

1. Se registra **`exp_002b_sarima_sutil_simple`** con SARIMA(1,1,1)(1,1,0,12) y
   horizonte corregido. `exp_002_sarima_sutil` se conserva sin borrar, con sus
   métricas recalculadas.
2. Se corrige el bug de horizonte en `02_gc1_sarima_prophet.ipynb` y se re-ejecuta
   para regenerar `predicciones.csv` y `metricas.json` de ambos SARIMA.
3. Para GC2 en adelante (SARIMAX-LSTM, GE, GM) se añade `t_index` como regresor de
   tendencia y se evalúa con lags alimentados por valores reales (one-step-ahead).
   Ver `DECISIONES_METODOLOGICAS.md`.

## Limitación reconocida

Los dos años de shock más severos de la serie (2017 y 2023) están **concentrados en
train**, y ni val (2024) ni test (2025) contienen un shock de esa magnitud. Las
métricas de test son, por tanto, optimistas respecto al desempeño esperable en un año
de shock. El split no se modifica —2025 es el holdout natural— pero la asimetría
queda declarada.